<a href="https://colab.research.google.com/github/WeizmannMLcourse/MLCourse_2025/blob/main/Tutorial9_GNN/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline

# This nb is modified from PyTorch Geometric documentation
- https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html
- https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_gnn.html \
With some additional material...

# Installation

Installing PyTorch Geometric can require matching your PyTorch version. This is what worked for me on:

**Colab** (i.e. Linux)

- CPU-only
```
pip uninstall -y torch torchvision torchaudio torchdata
pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cpu
pip install torch-geometric
```

- with CUDA
```
pip uninstall -y torch torchvision torchaudio torchdata
pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124
pip install torch-geometric
```

**macOS:**
```
conda create -n pyg-env python=3.10 -y
conda activate pyg-env
pip install torch torchvision packaging torch-geometric
```

In [ ]:
! pip uninstall -y torch torchvision torchaudio torchdata
! pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cpu
! pip install torch-geometric


How Does PyTorch Geometric Represent A Graph?
==============================================

By the end of this tutorial you will be able to:

-  Construct a graph in PyG from scratch.
-  Assign node and edge features to a graph.
-  Query properties of a graph such as node degrees and
   connectivity.
-  Transform a graph into another graph.
-  Work with graph objects built for message passing.

(Time estimate: 16 minutes)


PyG Graph Construction
----------------------

PyG represents a graph as a `Data` object.
You can construct a graph by specifying source and destination nodes
in `edge_index` with shape `[2, num_edges]`. Nodes in the graph have
consecutive IDs starting from 0.

For instance, the following code constructs a directed star graph with 5
leaves. The center node's ID is 0. The edges go from the
center node to the leaves.




In [ ]:
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.utils import to_networkx, to_undirected, degree

edge_index = torch.tensor([[0, 0, 0, 0, 0], [1, 2, 3, 4, 5]], dtype=torch.long)
g = Data(edge_index=edge_index, num_nodes=6)

edge_index = torch.stack([torch.LongTensor([0, 0, 0, 0, 0]), torch.LongTensor([1, 2, 3, 4, 5])])
g = Data(edge_index=edge_index, num_nodes=6)

g = Data(edge_index=torch.tensor([[0, 0, 0, 0, 0], [1, 2, 3, 4, 5]], dtype=torch.long))

Edges in the graph have consecutive IDs starting from 0, and are
in the same order as the list of source and destination nodes during
creation.




In [ ]:
# Print the source and destination nodes of every edge.
print(g.edge_index)

<div class="alert alert-info"><h4>Note</h4><p>Graphs are directed by default in `edge_index` to match
   graph neural network message passing, where messages
   from one node to another may differ across directions.
   For undirected graphs, you can symmetrize edges with
   `to_undirected`.</p></div>




### We can add and remove edges

In [ ]:
g1 = Data(edge_index=torch.tensor([[0, 0, 0, 0, 0], [1, 2, 3, 4, 5]], dtype=torch.long), num_nodes=6)
new_edges = torch.tensor([[1, 2, 2], [0, 0, 5]], dtype=torch.long)
g1.edge_index = torch.cat([g1.edge_index, new_edges], dim=1)
print(g1.edge_index)

g2 = Data(edge_index=torch.tensor([[0, 0, 0, 0, 0], [1, 2, 3, 4, 5]], dtype=torch.long), num_nodes=6)
keep_mask = torch.ones(g2.edge_index.shape[1], dtype=torch.bool)
keep_mask[[0, 1]] = False
g2.edge_index = g2.edge_index[:, keep_mask]
print(g2.edge_index)

### We can visualize the graph using `networkx`

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, g_ in enumerate([g, g1, g2]):
    nx.draw(to_networkx(g_, to_undirected=False), with_labels=True, ax=axes[i])

### We can also convert a graph to a bidirectional one

In [ ]:
g.edge_index = to_undirected(g.edge_index)

Assigning Node and Edge Features to Graph
-----------------------------------------

Many graph data contain attributes on nodes and edges.
Node and edge attributes are stored as tensors with matching first dimension
for the corresponding nodes or edges. In deep learning, those
attributes are often called *features*.

You can assign and retrieve node and edge features as attributes like `x`
and `edge_attr` on a `Data` object.




In [ ]:
# Assign a 3-dimensional node feature vector for each node.
g.x = torch.randn(6, 3)
# Assign a 4-dimensional edge feature vector for each edge.
g.edge_attr = torch.randn(g.edge_index.shape[1], 4)
# Assign a 5x4 node feature matrix for each node.
g.y = torch.randn(6, 5, 4)

print(g.edge_attr)

<div class="alert alert-info"><h4>Note</h4><p>The vast development of deep learning has provided us many
   ways to encode various types of attributes into numerical features.
   Here are some general suggestions:

   -  For categorical attributes (e.g. gender, occupation), consider
      converting them to integers or one-hot encoding.
   -  For variable length string contents (e.g. news article, quote),
      consider applying a language model.
   -  For images, consider applying a vision model such as CNNs.

   You can find plenty of materials on how to encode such attributes
   into a tensor in the `PyTorch Deep Learning
   Tutorials <https://pytorch.org/tutorials/>`__.</p></div>




Querying Graph Structures
-------------------------

`Data` objects provide direct access to graph structure through attributes.




In [ ]:
print(g.num_nodes)
print(g.edge_index.shape[1])
# Out degree of the center node
print(degree(g.edge_index[0], num_nodes=g.num_nodes)[0])
# In degree of the center node - note that the graph is undirected so the in and out degrees should match
print(degree(g.edge_index[1], num_nodes=g.num_nodes)[0])

# Message passing

<img src="https://github.com/WeizmannMLcourse/MLCourse_2025/blob/main/Tutorial8_GNN/messages.jpeg?raw=1"  width="500" height="665">

### In PyG, the same propagate/aggregate/update pattern is implemented with tensor operations

### First load some ones onto all the nodes

In [ ]:
ones = torch.ones(4)
g.count = ones.unsqueeze(0).repeat(g.num_nodes, 1)
print(g.count)

### Then for each node, let's add up its neighbors

In [ ]:
src, dst = g.edge_index
messages = g.count[src]
neighbor_sum = torch.zeros_like(g.count)
neighbor_sum.index_add_(0, dst, messages)
g.sum = neighbor_sum

### Let's look at the resulting feature

In [ ]:
print(g.sum)

### We can define our own custom aggregate function

In [ ]:
def my_function(messages, dst, num_nodes):
    my_sum = torch.zeros(num_nodes, messages.shape[1], dtype=messages.dtype)
    my_sum.index_add_(0, dst, messages)
    my_sum = my_sum / num_nodes
    return my_sum

In [ ]:
src, dst = g.edge_index
messages = g.count[src]
g['my sum'] = my_function(messages, dst, g.num_nodes)

In [ ]:
print(g['my sum'])

### Similarly, we can create our own propagation function

In [ ]:
def propagate(x, edge_attr, edge_index):
    src = edge_index[0]
    return x[src], edge_attr

In [ ]:
def sum_count_times_a(messages, edge_messages, dst, num_nodes):
    my_sum = torch.zeros(num_nodes, messages.shape[1], dtype=messages.dtype)
    my_sum.index_add_(0, dst, messages * edge_messages)
    return my_sum

In [ ]:
m, a = propagate(g.count, g.edge_attr, g.edge_index)
dst = g.edge_index[1]
g['my sum'] = sum_count_times_a(m, a, dst, g.num_nodes)

In [ ]:
print(g['my sum'])